# Map of Italian Science — Country & Organisation Citation Analysis

## Overview

This notebook investigates the **structural composition** of international citation networks for six Italian research universities: UNIBO, UNIMI, UNIPD, UNITO, UPO, and SNS. The central question is *how those citations are distributed* across the country's institutions.

A country with 50,000 citations could represent two entirely different realities: three flagship universities accounting for 90% of those citations — a fragile, hub-and-spoke structure — or hundreds of institutions each contributing a small share, indicating a resilient and systemically integrated network. Distinguishing between these two cases is the analytical task of this notebook.

The analysis moves through two scopes:

1. **First step analysis** — a static snapshot of concentration structure across all years combined, establishing the baseline structural archetypes.
2. **Temporal analysis** — how the structure evolves over time, from early bilateral ties toward systemic integration.

---

## Research Questions

**RQ1 — Citation Concentration:**
For each partner country, what percentage of its total citations to a given Italian institution is generated by its top-3 organisations? This reveals whether a country's citation volume is driven by a few dominant institutions or distributed across a broad network.

**RQ2 — Directional Asymmetry:**
Does the structural concentration of *incoming* citations (citing entities) match the concentration of *outgoing* citations (cited entitites)?

---

## Metrics

### Concentration Ratio — CR₃

The primary metric is the Concentration Ratio at N=3, which measures what fraction of a country's total citations to a given Italian institution come from that country's top-3 most-cited partner organisations:

$$CR_3(c) = \frac{\sum_{k=1}^{3} \text{citations from top-}k\text{ org in country }c}{\text{total citations from country }c} \times 100$$

- **CR₃ = 100%** → all citations come from a single organisation (maximum concentration)
- **CR₃ → 0%** → citations spread evenly across many organisations (maximum fragmentation)

### Herfindahl–Hirschman Index — HHI

The temporal analysis additionally uses the HHI, which considers the *entire* organisation distribution rather than just the top three, penalising dominant institutions quadratically:

$$HHI(c) = \sum_{i} \left(\frac{\text{citations from org}_i}{\text{total citations from country }c} \times 100\right)^2$$

An HHI near 10,000 means one institution monopolises the relationship; near 0 means perfect distribution. The HHI is used in Part II to confirm whether the concentration patterns identified by CR₃ hold at the full-distribution level and to surface non-monotonic trajectories that CR₃ alone cannot detect.

---

## Data

**Institutions:** UNIBO · UNIMI · UNIPD · UNITO · UPO · SNS

**Files per institution (aggregate):** `citation_counts_organizations_incoming.csv` / `citation_counts_organizations_outgoing.csv`

**Files per institution (temporal):** same files disaggregated into 5-year blocks under `citation_counts_annualized/`

**Key columns:** `country_name`, `country_code` (organisation), `count`

---

## Notebook Structure

> **Part I — Aggregate Concentration Analysis**
> - 1.1 Setup & data loading
> - 1.2 CR₃ calculation engine
> - 1.3 Scatter plot: citation volume vs. concentration
> - 1.4 Findings
>
> **Part II — Temporal Concentration Analysis (five 5-year blocks)**
> - 2.1 Setup & temporal data loading
> - 2.2 Scatter plot: how the volume–concentration relationship evolves
> - 2.3 Findings — 25-year network evolution
> - 2.4 HHI heatmap: full-distribution concentration over time
> - 2.5 Findings — HHI structural concentration
>
> **Appendix**
> Threshold diagnostic for MIN_CITATIONS


---
# Part I — Aggregate Concentration Analysis

## 1.1 Setup & Data Loading

The block below imports libraries and defines the two path constants that the rest of Part I depends on. `BASE_PATH` points to the **aggregate** organisation-level CSVs (all years combined). The six institution codes map to their full names via `INST_LABELS`, which every plot uses for readable titles.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import sys
import ipywidgets as widgets
from IPython.display import display, clear_output

# Paths & Environment
try:
    CURRENT_DIR = Path(__file__).resolve().parent
except NameError:
    CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "data_viz":
    sys.path.append(str(CURRENT_DIR.parent))
else:
    sys.path.append(str(CURRENT_DIR))

# Project imports
from src.data_utils import (
    INSTITUTIONS,
    INSTITUTION_LABELS,
    VISUALIZATIONS_PATH,
    load_org_data,
    load_all_available,
    load_all_temporal,
)
from src.validation import (
    export_cleaned_csvs,
    export_cleaned_csvs_temporal,
)

# Visualization settings
INST_COLORS = {
    "UNIBO": "#264653", "UNIMI": "#2a9d8f", "UNIPD": "#8ab17d",
    "UNITO": "#e9c46a", "UPO":   "#f4a261", "SNS":   "#e76f51",
}
DIR_COLORS = {"incoming": "#B7990D", "outgoing": "#320E3B"}
INST_LABELS = INSTITUTION_LABELS
DIR_LABELS  = {"incoming": "Incoming citations", "outgoing": "Outgoing citations"}


## 1.2 CR_N Calculation Engine

`calculate_concentration_metrics()` takes the combined master dataframe, a citation direction (`incoming` / `outgoing`), and a tier N, then returns one row per (institution × country) with:

- `total_country_citations` — the country's aggregate citation volume
- `top_n_citations` — sum of citations from the top-N organisations in that country  
- `CR_N` — the concentration ratio as a percentage

The loading function `load_and_calculate_concentration()` reads all available CSVs across the six institutions, combines them, and computes CR₃ for both directions. The function is tolerant of missing files (it silently skips them), so the notebook runs even on partial datasets.


In [ ]:
# CR₃ calculation (aggregate)

def calculate_concentration_metrics(df, direction='incoming', n=3):
    df_filtered = df[df['direction'] == direction].copy()
    if df_filtered.empty:
        return pd.DataFrame()

    weight_col = (
        'counts' if 'counts' in df_filtered.columns else
        'count'  if 'count'  in df_filtered.columns else
        'derived_counts'
    )
    if weight_col == 'derived_counts':
        df_filtered['derived_counts'] = 1

    country_totals = (
        df_filtered
        .groupby(['italian_institution', 'country_name'])[weight_col]
        .sum().reset_index(name='total_country_citations')
    )
    org_totals = (
        df_filtered
        .groupby(['italian_institution', 'country_name', 'legal_name'])[weight_col]
        .sum().reset_index(name='org_citations')
    )
    org_totals = org_totals.sort_values(
        ['italian_institution', 'country_name', 'org_citations'],
        ascending=[True, True, False]
    )
    top_n_orgs = org_totals.groupby(['italian_institution', 'country_name']).head(n)
    top_n_sums = (
        top_n_orgs
        .groupby(['italian_institution', 'country_name'])['org_citations']
        .sum().reset_index(name='top_n_citations')
    )
    cr_df = pd.merge(country_totals, top_n_sums,
                     on=['italian_institution', 'country_name'], how='left')
    cr_df['top_n_citations'] = cr_df['top_n_citations'].fillna(
        cr_df['total_country_citations']
    )
    cr_df[f'CR_{n}'] = (
        cr_df['top_n_citations'] / cr_df['total_country_citations']
    ) * 100

    return cr_df.sort_values(
        by=['italian_institution', 'total_country_citations'],
        ascending=[True, False]
    )

datasets = load_all_available()

if datasets:
    agg_master = pd.concat(
        [
            df.assign(italian_institution=inst)
            for inst, (inc, out) in datasets.items()
            for df in (inc, out)
        ],
        ignore_index=True,
    )
    static_incoming_cr_df = calculate_concentration_metrics(agg_master, 'incoming', 3)
    static_outgoing_cr_df = calculate_concentration_metrics(agg_master, 'outgoing', 3)
else:
    static_incoming_cr_df = None
    static_outgoing_cr_df = None

agg_incoming_cr_df = static_incoming_cr_df
agg_outgoing_cr_df = static_outgoing_cr_df

In [ ]:
# Export cleaned aggregate org CSVs
export_cleaned_csvs(
    loader_function=load_org_data,
    filename_prefix="organizations",
    output_dir=VISUALIZATIONS_PATH.parent, # Saves to /visualizations/
    institutions=INSTITUTIONS,
)

## 1.3 Scatter plot: Citation Volume vs. Concentration

`plot_single_inst` creates two bubble scatter plots per selected institution — one for incoming citations (papers that cite this institution), one for outgoing (papers this institution cites). Each bubble is a partner country. Position encodes the core tension of this analysis: the x-axis shows how much that country contributes in raw citations (log scale), while the y-axis shows how concentrated those citations are in its top-3 organisations.

**How to read it:**

| Quadrant | What it means |
|---|---|
| Top-left: high CR_N, low volume | Niche or emerging partner — one or two foreign flagships drive most of the relationship |
| Top-right: high CR_N, high volume | Large science system but structurally dependent on a small elite |
| Bottom-left: low CR_N, low volume | Peripheral partner with broad but thin engagement |
| Bottom-right: low CR_N, high volume | Mature, deeply integrated partner — the ideal profile |

The **dashed line at CR_N = 50%** marks the "concentrated" threshold: above it, three organisations account for the majority of a country's citations. Use the dropdown to switch between institutions.

In [ ]:
# Build a persistent color map
def _build_country_color_map(*cr_dfs):
    all_countries = sorted(set(
        country
        for df in cr_dfs if df is not None
        for country in df['country_name'].unique()
    ))
    palette = px.colors.qualitative.Pastel + px.colors.qualitative.Set3
    return {country: palette[i % len(palette)] for i, country in enumerate(all_countries)}

COUNTRY_COLOR_MAP = _build_country_color_map(static_incoming_cr_df, static_outgoing_cr_df)


def plot_single_inst(inst_name, cr_df, direction, n=3):
    plot_df = (
        cr_df[cr_df['italian_institution'] == inst_name]
        .sort_values('total_country_citations', ascending=False)
        .head(20)
        .copy() 
    )
    
    label      = INST_LABELS.get(inst_name, inst_name)
    base_color = DIR_COLORS[direction]
    badge_text = '▼ Incoming' if direction == 'incoming' else '▲ Outgoing'

    fig = px.scatter(
        plot_df,
        x='total_country_citations',
        y=f'CR_{n}',
        size='total_country_citations',
        color='country_name',
        text='country_name',
        log_x=True,
        template='plotly_white',
        height=550,
        color_discrete_map=COUNTRY_COLOR_MAP,
        labels={
            'total_country_citations': 'Total citations (log scale)',
            f'CR_{n}': f'CR{n} — top-{n} org concentration (%)',
            'country_name': 'Country',
        },
        title=(
            f'<b>Citation Concentration vs. Volume</b>  ·  {label}'
            f'  ·  {DIR_LABELS[direction]}'
        ),
    )
    fig.update_traces(textposition='top center', marker=dict(opacity=0.82))

    # Direction badge
    fig.add_annotation(
        xref='paper', yref='paper', x=0.0, y=1.07,
        text=(
            f'<span style="background:{base_color};color:white;'
            f'padding:2px 8px;border-radius:4px;font-size:12px">'
            f'{badge_text}</span>'
        ),
        showarrow=False, xanchor='left',
    )

    fig.add_hline(y=50, line_dash='dash', line_color=base_color, opacity=0.5)
    fig.add_annotation(
        text='50% concentration threshold',
        xref='paper', x=1.01, yref='y', y=50,
        xanchor='left', showarrow=False,
        font=dict(size=11, color=base_color),
    )
    fig.update_yaxes(range=[0, 115], title=f'CR{n} (%)')
    fig.update_layout(showlegend=True, margin=dict(r=160))
    return fig

if static_incoming_cr_df is not None:
    out_static = widgets.Output()

    inst_static_dd = widgets.Dropdown(
        options=[(INST_LABELS[i], i) for i in INSTITUTIONS],
        value='UNIBO',
        description='Institution:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px'),
    )
    dir_static_dd = widgets.Dropdown(
        options=[('Incoming citations', 'incoming'), ('Outgoing citations', 'outgoing')],
        value='incoming',
        description='Direction:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='230px'),
    )
    static_controls = widgets.HBox(
        [inst_static_dd, dir_static_dd],
        layout=widgets.Layout(gap='20px', margin='0 0 12px 0'),
    )

    def render_static(inst_name, direction):
        cr_df = static_incoming_cr_df if direction == 'incoming' else static_outgoing_cr_df
        with out_static:
            clear_output(wait=True)
            plot_single_inst(inst_name, cr_df, direction).show()

    def on_static_change(change):
        render_static(inst_static_dd.value, dir_static_dd.value)

    inst_static_dd.observe(on_static_change, names='value')
    dir_static_dd.observe(on_static_change, names='value')

    display(static_controls, out_static)
    render_static('UNIBO', 'incoming')

else:
    print("No data loaded — check BASE_PATH and file names.")

## 1.4 Findings

The scatter plots reveal a remarkably consistent structural picture across all six institutions and both citation directions. The core geometry is the same in every view: a dense cluster of major partners in the bottom-right (high volume, low concentration), a dispersed scatter of smaller partners in the left half of the chart, and an almost empty upper half of the plot. This last point is the most important finding: **with very few exceptions, no country in the top-20 by volume exceeds the 50% concentration threshold**. The concentrated zone is essentially vacant for all Italian institutions; the threshold becomes relevant only for UPO and SNS, which operate at smaller scale.

---

### The concentrated zone is almost empty

Across UNIBO, UNIMI, UNIPD, and UNITO — in both directions — not a single country in the top-20 by citation volume crosses or even approaches the 50% line. Every major partner sits well below 40% CR₃, and most of the high-volume cluster sits below 20%. This is not a mild version of fragmentation: it is near-complete absence of elite capture for the four largest institutions. For these universities, the question of "which three organisations dominate a country's citations" is analytically moot for every partner that matters by volume.

The picture changes only at UPO and SNS, where the smaller overall citation base means some countries appear above the threshold:

- **Finland** is the only country that consistently exceeds 50% CR₃ across multiple institutions and directions — visible in UPO incoming (~60%), UPO outgoing (~60%), SNS outgoing (~62%), and SNS incoming (~63%). At these smaller institutions, Finnish citations are effectively controlled by one or two organisations, almost certainly a single leading research university with a specific disciplinary link.
- **Switzerland** sits near or just above the 50% line in both SNS directions (~52–53%), consistent with concentrated elite engagement — likely the ETH domain — that has not yet diversified into the broader Swiss science system.
- **Belgium** (~52%) and **Hungary** (outgoing only) appear above the threshold exclusively at SNS, reflecting the institution's narrow disciplinary scope generating highly concentrated relationships with specific foreign partners.

The implication is clear: concentration risk is not a system-wide problem for Italian international citation networks. It is institution-specific and scale-dependent, concentrated at the two smallest institutions.


### The fragmented core: US, France, Germany, UK

These four countries occupy the extreme bottom-right in every single view. Their CR₃ values stay consistently below 15–20% regardless of institution or direction. Notably, this holds even for UPO and SNS despite their smaller scale — the structural fragmentation of engagement with the US, France, Germany, and the UK is a property of those countries' science systems, not of the Italian institution's size or profile.

The US is always the rightmost bubble by a substantial margin, with CR₃ near or below 5% everywhere. At UNIBO and UNIMI it reaches 2–4M citations while maintaining this near-zero concentration. This means that even taken together, the top-3 American institutions account for fewer than 5% of all US citations to these universities — a level of distribution that is essentially immune to any single institutional fluctuation.


### Italy's structural anomaly

Italy is the most analytically interesting outlier in the dataset. It consistently appears as a mid-field bubble — high volume but with CR₃ notably above the trend line for its size, typically in the 25–35% range across all institutions and both directions. This places it structurally closer to mid-tier partners like Switzerland or Brazil than to the major international science systems it sits alongside in terms of raw volume.

This is not an international dependency — it is a domestic structural feature. Cross-institutional citations within Italy are concentrated in the same few leading universities that appear at the top of Italian research output rankings. The Italian academic system generates a specific citation pattern: high volume but concentrated through a small number of research universities, which is precisely what a high-but-not-extreme CR₃ captures.


### Incoming vs. outgoing: structural symmetry with directional nuance

The most striking feature when switching directions is how similar the charts are. Country rankings, CR₃ values, and the overall geometry are preserved in almost every case. The main consistent difference is **volume**: outgoing citation counts are systematically lower than incoming across all institutions, which shifts the x-axis scale leftward. This reflects the general pattern that these Italian institutions receive more international citations than they generate — their output is internationally cited by a broad base, while their own reference lists, though broad, are somewhat more selective.

Beyond the volume shift, a few direction-specific differences are visible:

**SNS** shows the strongest directional divergence. In the outgoing chart, Spain sits at ~35% CR₃ — noticeably higher than its ~25% in the incoming view — and Hungary appears in the top-20 outgoing but not incoming. This is consistent with SNS's highly specialised disciplinary profile: its researchers cite a narrower, more concentrated set of foreign partners than the international community that cites SNS output. The outgoing network is more selective; the incoming network is broader.

**UNITO incoming** is the only chart where Taiwan appears in the top-20, suggesting a specific disciplinary import relationship (likely physics or mathematics) that does not manifest symmetrically in outgoing citations.

**UNIMI incoming** shows Sweden and Denmark near the threshold (~47–50%), while these countries sit lower in the outgoing view. UNIMI's incoming citations from Scandinavian countries appear to be driven by a handful of institutions, possibly in the life sciences.

**UPO** shows the highest directional symmetry of all six institutions — the charts are nearly identical in both directions, which is consistent with UPO being a smaller, more specialised university whose citation relationships are stable and bidirectional.

---

### Structural implications

The aggregate snapshot delivers three clear conclusions. First, concentration risk is effectively absent for the four largest institutions — the relevant policy question for UNIBO, UNIMI, UNIPD, and UNITO is not "are we over-reliant on a few foreign organisations?" but "how broadly are our international relationships distributed and growing?". Second, for UPO and SNS, Finland and Switzerland represent the only genuine structural concentrations worth monitoring, though even these may reflect disciplinary alignment rather than dependency. Third, Italy's anomalous mid-field position is a persistent domestic structural feature that deserves separate treatment in any analysis of citation network resilience.

---
# Part II — Temporal Concentration Analysis (five 5-year blocks)

This section adds the time dimension. Instead of a single aggregate snapshot, citation data is disaggregated into five consecutive 5-year blocks: 2001–2005, 2006–2010, 2011–2015, 2016–2020, and 2021–2025. The question shifts from *what is the structure?* to *how does the structure evolve?*

## 2.1 Setup & Temporal Data Loading

`ANNUALIZED_PATH` points to the temporal CSV files. `YEAR_BLOCKS` defines the five periods. All other constants (`INSTITUTIONS`, `INST_LABELS`, `DIR_COLORS`) are inherited from the Part I setup cell. The loader returns `master_df` alongside the pre-computed CR₃ dataframes — `master_df` is kept as a global because the HHI calculation in Section 2.4 operates directly on the raw organisation-level data.

In [ ]:
# Part II configuration
YEAR_BLOCKS = ['2001-2005', '2006-2010', '2011-2015', '2016-2020', '2021-2025']


# Temporal CR₃ calculation
def calculate_concentration_metrics_temporal(df, direction='incoming', n=3):
    df_filtered = df[df['direction'] == direction].copy()
    if df_filtered.empty:
        return pd.DataFrame()

    weight_col = (
        'counts' if 'counts' in df_filtered.columns else
        'count'  if 'count'  in df_filtered.columns else
        'derived_counts'
    )
    if weight_col == 'derived_counts':
        df_filtered['derived_counts'] = 1

    country_totals = (
        df_filtered
        .groupby(['italian_institution', 'year_block', 'country_name'])[weight_col]
        .sum().reset_index(name='total_country_citations')
    )
    
    org_totals = (
        df_filtered
        .groupby(['italian_institution', 'year_block', 'country_name', 'legal_name'])[weight_col]
        .sum().reset_index(name='org_citations')
    )
    org_totals = org_totals.sort_values(
        ['italian_institution', 'year_block', 'country_name', 'org_citations'],
        ascending=[True, True, True, False]
    )
    top_n_orgs = org_totals.groupby(
        ['italian_institution', 'year_block', 'country_name']
    ).head(n)
    top_n_sums = (
        top_n_orgs
        .groupby(['italian_institution', 'year_block', 'country_name'])['org_citations']
        .sum().reset_index(name='top_n_citations')
    )
    cr_df = pd.merge(country_totals, top_n_sums,
                     on=['italian_institution', 'year_block', 'country_name'], how='left')
    cr_df['top_n_citations'] = cr_df['top_n_citations'].fillna(
        cr_df['total_country_citations']
    )
    cr_df[f'CR_{n}'] = (
        cr_df['top_n_citations'] / cr_df['total_country_citations']
    ) * 100

    return cr_df.sort_values(
        by=['italian_institution', 'year_block', 'total_country_citations'],
        ascending=[True, True, False]
    )

def load_temporal_data():
    master_df = load_all_temporal(dataset_type='organizations')

    if master_df.empty:
        print("No annualized org files found — check ANNUALIZED_BASE_PATH in data_utils.")
        return None, None, None

    if 'institution' in master_df.columns and 'italian_institution' not in master_df.columns:
        master_df = master_df.rename(columns={'institution': 'italian_institution'})

    return (
        master_df,
        calculate_concentration_metrics_temporal(master_df, 'incoming', 3),
        calculate_concentration_metrics_temporal(master_df, 'outgoing', 3),
    )


master_df, incoming_cr_df, outgoing_cr_df = load_temporal_data()

In [ ]:
# Export cleaned temporal CSVs (Both Orgs and Countries!)

export_cleaned_csvs_temporal(
    output_dir=VISUALIZATIONS_PATH,
)

## 2.2 Scatter Plot: Volume vs. Concentration Over Time

The following cell displays the same bubble scatter plot from Part I, but animated across the five time blocks. Each frame is one 5-year period. Use the ▶ play button or drag the slider to move through time. The Institution and Direction dropdowns control what is shown.

**What to look for as the animation plays:**

1. **Rightward drift** — bubbles moving right signal citation volume growth. Nearly all countries drift right; the speed separates rapidly emerging partners from stable ones.
2. **Downward drift** — bubbles moving down mean a broader set of the country's institutions is engaging. This is the structural signature of system-wide integration replacing bilateral partnerships.
3. **Crossing the 50% line** — the moment a country drops below the dashed threshold marks its transition from a niche to a systemic relationship.
4. **Outlier persistence** — countries that stay above CR₃ = 50% across multiple periods maintain structural dependencies regardless of volume growth.
5. **The 2021–2025 pullback** — a slight leftward movement in the final block is visible across institutions. This is a citation lag artefact: papers published in 2023–2025 have had less time to accumulate citations, so the final block undercounts volume relative to earlier ones.

In [ ]:
# VISUALIZATION ENGINE
def plot_single_inst_animated_v2(inst_name, cr_df, direction, n=3):
    # Filter to this institution
    plot_df = cr_df[cr_df['italian_institution'] == inst_name].copy()
    plot_df = plot_df.sort_values('year_block')
 
    # Keep the top-25 countries by peak volume
    top_countries = (
        plot_df.groupby('country_name')['total_country_citations']
               .max()
               .nlargest(25)
               .index
    )
    plot_df = plot_df[plot_df['country_name'].isin(top_countries)].copy()
 
    # Decide which countries get permanent text labels
    # Only the top-10 by peak volume; others appear in hover only.
    top10_label_countries = (
        plot_df.groupby('country_name')['total_country_citations']
               .max()
               .nlargest(10)
               .index
               .tolist()
    )
    plot_df['label_text'] = plot_df['country_name'].where(
        plot_df['country_name'].isin(top10_label_countries), other=''
    )
 
    min_x = max(500,  plot_df['total_country_citations'].min() * 0.4)
    max_x =           plot_df['total_country_citations'].max() * 2.0
 
    cr_col = f'CR_{n}'
    direction_label = 'Incoming' if direction == 'incoming' else 'Outgoing'
 
    base_color = DIR_COLORS[direction]

    fig = px.scatter(
        plot_df,
        x='total_country_citations',
        y=cr_col,
        size='total_country_citations',
        color='country_name',
        text='label_text',
        log_x=True,
        template='plotly_white',
        height=620,
        size_max=55,
        color_discrete_sequence=px.colors.qualitative.Pastel,
        animation_frame='year_block',
        animation_group='country_name',
        range_x=[min_x, max_x],
        range_y=[0, 110],
        custom_data=['country_name', 'year_block', 'total_country_citations', cr_col],
        title=(
            f'<b>Citation Concentration vs. Volume</b>  ·  '
            f'{INST_LABELS.get(inst_name, inst_name)}  ·  {direction_label} Citations'
        ),
    )
 
    # Hover template
    fig.update_traces(
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Period: %{customdata[1]}<br>'
            'Total citations: %{customdata[2]:,.0f}<br>'
            f'CR₃ (top-3 share): %{{customdata[3]:.1f}}%'
            '<extra></extra>'
        ),
        textposition='top center',
        marker=dict(opacity=0.85, sizemode='area'),
    )
 
    # 50% line — "concentrated" threshold
    fig.add_hline(y=50, line_dash='dash', line_color=base_color, opacity=0.55)
    fig.add_annotation(
        xref='paper', x=0.01,
        yref='y',     y=50,
        text='Concentrated (CR₃ = 50%)',
        showarrow=False,
        xanchor='left',
        yanchor='bottom',
        font=dict(size=11, color=base_color),
        bgcolor='rgba(255,255,255,0.7)',
    )

    # 25% line — "moderate" threshold
    fig.add_hline(y=25, line_dash='dot', line_color=base_color, opacity=0.35)
    fig.add_annotation(
        xref='paper', x=0.01,
        yref='y',     y=25,
        text='Moderate (CR₃ = 25%)',
        showarrow=False,
        xanchor='left',
        yanchor='bottom',
        font=dict(size=11, color=base_color),
        bgcolor='rgba(255,255,255,0.7)',
    )
 
    # Axis formatting
    fig.update_xaxes(
        title_text='Total citation volume (log scale)',
        title_font_size=13,
        tickfont_size=11,
        tickvals=[1_000, 2_000, 5_000, 10_000, 20_000, 50_000,
                  100_000, 200_000, 500_000, 1_000_000, 2_000_000],
        ticktext=['1k', '2k', '5k', '10k', '20k', '50k',
                  '100k', '200k', '500k', '1M', '2M'],
        showgrid=True, gridcolor='#ebebeb',
    )
    fig.update_yaxes(
        title_text='CR₃ — share of citations from top-3 organisations (%)',
        title_font_size=13,
        tickfont_size=11,
        ticksuffix='%',
        showgrid=True, gridcolor='#ebebeb',
        zeroline=False,
    )
 
    # Layout polish
    fig.update_layout(
        title_font_size=15,
        title_x=0.0,
        legend=dict(
            title_text='',
            x=1.01, y=1,
            xanchor='left', yanchor='top',
            font_size=11,
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='#ddd',
            borderwidth=1,
        ),
        margin=dict(l=70, r=200, t=70, b=60),
        transition={'duration': 400},
        sliders=[{
            'currentvalue': {
                'prefix': 'Period: ',
                'font': {'size': 13},
            },
        }],
    )
 
    return fig

# INTERACTIVE DASHBOARD
if incoming_cr_df is not None and not incoming_cr_df.empty:

    out_anim = widgets.Output()

    inst_anim_dd = widgets.Dropdown(
        options=[(INST_LABELS[i], i) for i in INSTITUTIONS],
        value='UNIBO',
        description='Institution:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='280px'),
    )
    dir_anim_dd = widgets.Dropdown(
        options=[('Incoming citations', 'incoming'), ('Outgoing citations', 'outgoing')],
        value='incoming',
        description='Direction:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='230px'),
    )

    anim_controls = widgets.HBox(
        [inst_anim_dd, dir_anim_dd],
        layout=widgets.Layout(gap='20px', margin='0 0 15px 0'),
    )

    def render_animated(inst_name, direction):
        cr_data = calculate_concentration_metrics_temporal(master_df, direction, 3)
        with out_anim:
            clear_output(wait=True)
            if not cr_data.empty:
                plot_single_inst_animated_v2(inst_name, cr_data, direction, 3).show()

    def on_anim_change(change):
        render_animated(inst_anim_dd.value, dir_anim_dd.value)

    inst_anim_dd.observe(on_anim_change, names='value')
    dir_anim_dd.observe(on_anim_change, names='value')

    display(anim_controls, out_anim)
    render_animated('UNIBO', 'incoming')

else:
    print("No data loaded — check BASE_PATH and file names.")

## 2.3 Findings — 25-Year Network Evolution

### The dominant macro-trend: right and down

Across all six institutions and both directions, the movement from 2001–2005 to 2021–2025 is rightward (volume growth) combined with downward (falling CR₃). This is a structural property of citation network growth: as the total number of citations from a country increases, the probability that any single organisation maintains a dominant share decreases mechanically, because new citations distribute across an ever-broader institutional base.

### Trajectories of Key Global Partners

**The Stable Anchor — United States.**  
The US occupies the extreme bottom-right from the earliest time block onward. Its CR₃ remains below 10% throughout, confirming that even in 2001–2005 the American science system was too large and diverse to concentrate in a few institutions. Its bubble grows rightward every period; its structural position barely changes.

**The Rapid Emergence — China.**  
In 2001–2005, China appears as a mid-tier partner with moderate volume and a CR₃ near 20–30%. By 2021–2025 it has accelerated into the highest-volume tier alongside the US and major European partners, while its CR₃ has dropped steeply. This arc tracks the structural maturation of China's research system: from a handful of early-adopter institutions (leading universities engaging internationally before the system-wide expansion) to a broad national network.

**The European Core — UK, Germany, France, Spain.**  
These partners move as a coherent group: stable, low-concentration profiles (CR₃ < 20%) drifting steadily rightward. They never spend significant time above the 50% threshold. This reflects deep, pre-existing integration within the European Research Area rather than the emergence of new bilateral ties.

**India's Structural Transformation.**  
Among all partners, India shows the steepest concentration *drop*. In 2001–2005 its CR₃ hovers near 60–65%; by 2021–2025 it falls below 15%. This tracks the rapid expansion and diversification of India's university research base over the same period.

**Persistent Concentrations — Small Northern European Countries.**  
Finland, Denmark, and Sweden remain structurally concentrated relative to their volume across most of the 25 years. Their citation relationships with Italian institutions appear to be driven by specific bilateral partnerships rather than system-wide integration. This may reflect disciplinary specialisation.

### Directional symmetry

Switching direction from incoming to outgoing produces nearly identical charts for every institution. The country ranking, cluster geometry, and individual CR₃ values are all preserved. The only systematic difference is a slight compression of the x-axis — outgoing citation counts are typically 10–20% lower than incoming — reflecting the general pattern that these Italian institutions receive more citations than they generate relative to their largest partners.

## 2.4 HHI: Full-Distribution Concentration Index

CR₃ captures only the share of the top-3 organisations. The **Herfindahl–Hirschman Index (HHI)** considers the *entire* organisation distribution, penalising dominant institutions quadratically. It answers a question CR₃ cannot: is a country's concentration driven by one single institution, or is it spread across a small cluster of similarly-sized ones?

The heatmap shows the top-15 countries by mean HHI for the selected institution and direction. Countries with fewer than 2,000 peak citations are excluded — below this floor, HHI = 10,000 is almost always a trivial artefact of having only one organisation in the dataset. Countries are sorted top-to-bottom from most to least concentrated across the five blocks.

**HHI interpretation:** values above 2,500 indicate one or two dominant organisations; 1,000–2,500 is a moderate cluster; below 1,000 is broadly fragmented.

**How to read it alongside the animated scatter:** the scatter shows *where* a country sits in the volume–concentration space at a given moment. The heatmap shows *how stable* that position is over time. A row that fades from dark to light confirms a transition seen in the scatter animation. A row that spikes mid-period then recovers — invisible in the scatter — is the heatmap's distinctive contribution: it surfaces non-monotonic trajectories that only become apparent when all five periods are viewed simultaneously.

In [ ]:
YEAR_BLOCKS_HHI = ['2001-2005', '2006-2010', '2011-2015', '2016-2020', '2021-2025']
MIN_CITATIONS_HHI   = 2000
MIN_CR3_HHI         = 20.0
TOP_N_COUNTRIES_HHI = 15


def calculate_longitudinal_hhi(df, direction='incoming'):
    if df is None:
        return None
    df_filtered = df[df['direction'] == direction].dropna(
        subset=['country_name', 'legal_name']
    ).copy()

    weight_col = (
        'counts' if 'counts' in df_filtered.columns else
        'count'  if 'count'  in df_filtered.columns else
        'derived_counts'
    )
    if weight_col == 'derived_counts':
        df_filtered['derived_counts'] = 1

    country_sums = (
        df_filtered
        .groupby(['italian_institution', 'year_block', 'country_name'])[weight_col]
        .transform('sum')
    )
    df_filtered['share']         = (df_filtered[weight_col] / country_sums) * 100
    df_filtered['share_squared'] = df_filtered['share'] ** 2

    hhi_df = (
        df_filtered
        .groupby(['italian_institution', 'year_block', 'country_name'])['share_squared']
        .sum().reset_index(name='HHI')
    )
    vol_df = (
        df_filtered
        .groupby(['italian_institution', 'year_block', 'country_name'])[weight_col]
        .sum().reset_index(name='total_citations')
    )
    return hhi_df.merge(vol_df, on=['italian_institution', 'year_block', 'country_name'], how='left')


hhi_incoming_df = calculate_longitudinal_hhi(master_df, 'incoming')
hhi_outgoing_df = calculate_longitudinal_hhi(master_df, 'outgoing')


def plot_hhi_heatmap(inst_name, hhi_df, cr_df, direction):
    if hhi_df is None or (hasattr(hhi_df, 'empty') and hhi_df.empty):
        print("No temporal data — check ANNUALIZED_BASE_PATH in data_utils.")
        return go.Figure()

    inst_hhi = hhi_df[hhi_df['italian_institution'] == inst_name].copy()
    peak_vol  = inst_hhi.groupby('country_name')['total_citations'].max()
    volume_ok = peak_vol[peak_vol >= MIN_CITATIONS_HHI].index

    inst_cr = cr_df[cr_df['italian_institution'] == inst_name]
    cr_col  = [c for c in inst_cr.columns if c.startswith('CR_')][0]
    conc_ok = inst_cr[inst_cr[cr_col] >= MIN_CR3_HHI]['country_name'].unique()

    inst_df = inst_hhi[
        inst_hhi['country_name'].isin(volume_ok) &
        inst_hhi['country_name'].isin(conc_ok)
    ]
    if inst_df.empty:
        print(f"No countries meet both criteria for {inst_name} ({direction}).")
        return go.Figure()

    top_countries = (
        inst_df.groupby('country_name')['HHI']
        .mean().nlargest(TOP_N_COUNTRIES_HHI).index.tolist()
    )
    inst_df = inst_df[inst_df['country_name'].isin(top_countries)]

    pivot = (
        inst_df
        .pivot_table(index='country_name', columns='year_block', values='HHI', aggfunc='mean')
        .reindex(columns=YEAR_BLOCKS_HHI)
    )
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]
    z = pivot.values
    y_labels = pivot.index.tolist()

    hover = [
        [
            f"{y}<br>Period: {x}<br>HHI: {v:,.0f}" if not np.isnan(v)
            else f"{y}<br>Period: {x}<br>No data"
            for x, v in zip(YEAR_BLOCKS_HHI, row_vals)
        ]
        for y, row_vals in zip(y_labels, z)
    ]

    direction_label = 'Incoming' if direction == 'incoming' else 'Outgoing'
    label = INST_LABELS.get(inst_name, inst_name)
    colorscale = [
        [0.0, '#C9A84C'], [0.25, '#D4B96A'],
        [0.5, '#C5A8C5'], [0.75, '#8B6AB0'], [1.0, '#6B4E9B'],
    ]

    fig = go.Figure(go.Heatmap(
        z=z, x=YEAR_BLOCKS_HHI, y=y_labels, zmin=0, zmax=5000,
        colorscale=colorscale,
        colorbar=dict(
            title='HHI',
            tickvals=[0, 1000, 2500, 5000],
            ticktext=['0 (fragmented)', '1,000', '2,500', '5,000+ (concentrated)'],
            thickness=15, len=0.7,
        ),
        text=hover, hovertemplate='%{text}<extra></extra>',
        xgap=3, ygap=3,
    ))
    fig.update_layout(
        title=dict(
            text=(
                f'<b>HHI Full-Distribution Concentration — {label}</b><br>'
                f'<sup>{direction_label} · peak CR₃ ≥ {MIN_CR3_HHI:.0f}% · '
                f'≥ {MIN_CITATIONS_HHI:,} citations · ranked by mean HHI · '
                f'dark = concentrated · golden = fragmented</sup>'
            ),
            x=0.0, font_size=14,
        ),
        height=520, template='plotly_white',
        margin=dict(t=100, b=60, l=180, r=160),
        xaxis=dict(tickangle=30, tickfont_size=11, side='bottom'),
        yaxis=dict(tickfont_size=11),
    )
    return fig


out_hhi = widgets.Output()
inst_hhi_dd = widgets.Dropdown(
    options=[(INST_LABELS[i], i) for i in INSTITUTIONS], value='UNIBO',
    description='Institution:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)
dir_hhi_dd = widgets.Dropdown(
    options=[('Incoming citations', 'incoming'), ('Outgoing citations', 'outgoing')],
    value='incoming', description='Direction:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='240px'),
)
hhi_controls = widgets.HBox(
    [inst_hhi_dd, dir_hhi_dd],
    layout=widgets.Layout(gap='20px', margin='0 0 12px 0'),
)

def render_hhi(inst_name, direction):
    cr_df  = agg_incoming_cr_df if direction == 'incoming' else agg_outgoing_cr_df
    hhi_df = hhi_incoming_df    if direction == 'incoming' else hhi_outgoing_df
    with out_hhi:
        clear_output(wait=True)
        plot_hhi_heatmap(inst_name, hhi_df, cr_df, direction).show()

def on_hhi_change(change):
    render_hhi(inst_hhi_dd.value, dir_hhi_dd.value)

inst_hhi_dd.observe(on_hhi_change, names='value')
dir_hhi_dd.observe(on_hhi_change, names='value')
display(hhi_controls, out_hhi)
render_hhi('UNIBO', 'incoming')


## 2.5 Findings — HHI Structural Concentration

The HHI heatmap confirms the patterns identified by CR₃ and adds two layers: full-distribution validation (confirming concentration is real, not an artefact of the top-3 choice) and temporal resolution of non-monotonic cases.

**High-baseline partners and persistent dependencies** Cyprus, Bulgaria, Armenia, Lithuania: These countries appear in the top-15 across almost every institution and direction. However, rather than remaining structurally static, they demonstrate how long it takes to dilute early monopolies. Relationships with countries like Armenia and Lithuania began as total dependencies (HHI ≥ 5,000 in 2001–2010) and have steadily cooled into the 1,000–2,500 range. Conversely, Cyprus (visible at SNS Incoming) remains persistently in the high-risk zone (HHI 3,400 to 5,400) across the entire 25-year window, while Bulgaria (UPO Outgoing) starts as a total monopoly (HHI = 10,000) and stays highly concentrated (HHI > 5,000). These are the clearest candidates for deliberate partnership broadening if network resilience is a policy objective.

**Steep decentralisation** — Ecuador, Peru, Argentina, Ukraine, Georgia: These countries show the most dramatic dark-to-light fades, representing true systemic maturation. Ecuador’s arc is the most extreme (e.g., at UNIMI Incoming), starting as a pure monopoly (HHI = 10,000) in early blocks and dropping into highly fragmented territory (HHI < 150) by 2021–2025. Argentina, Georgia, and Peru (highly visible at UPO Outgoing and SNS Incoming) follow similar, rapid trajectories from total monopoly down to the sub-1,000 range, perfectly tracking the expansion and integration of their domestic higher education systems.

**Non-monotonic spikes** - Serbia, Azerbaijan, Latvia, Puerto Rico: These countries show concentration increasing in a middle block before decreasing. Serbia (at SNS Incoming) is the textbook example: starting fragmented (HHI = 1,100), spiking massively in 2006–2010 (HHI = 3,444), and completely dissipating back into fragmentation by 2025 (HHI = 529). This pattern is inconsistent with gradual maturation and strongly suggests a specific, intense bilateral programme that flared up and then concluded. The heatmap makes this structural dynamic instantly visible; an animated scatter plot, which shows only one moment at a time, cannot.

**Institution-specific signals**: Each institution displays country-specific anomalies reflecting distinct disciplinary or strategic profiles. Examples include Uruguay at UNIBO (Incoming), Jordan at UNIMI (Outgoing), Iceland at both UNIPD and UNITO, Colombia and Bulgaria at UPO (Outgoing), and Thailand persistently concentrated at SNS (Outgoing). These institutional outliers are prime indicators of specialised bilateral agreements or hyper-specific research niches.

**Directional asymmetry**: For most countries, the HHI profile is broadly symmetric between incoming and outgoing flows, confirming bidirectional maturation. However, SNS shows strong asymmetries (e.g., highly concentrated outgoing flows to places like Thailand and Bulgaria), consistent with its highly specialised profile: its output is cited by a narrower, specific international community, even if its own reference lists draw from a broader base.

---
## Appendix — Threshold Diagnostic: Choosing `MIN_CITATIONS`

The `MIN_CITATIONS` filter in the HHI heatmap excludes countries whose peak citation volume across all blocks is below the threshold. The goal is to remove micro-states and peripheral partners where HHI = 10,000 simply because one institution sent one paper — not because the relationship is structurally concentrated.

**How to read the output below:**
- The percentile table shows how many countries survive at each candidate threshold. A good threshold sits just above the "long tail" of near-zero-volume countries, keeping all partners with genuine engagement.
- The scatterplot of peak volume vs. peak HHI reveals whether the two are correlated at low volumes (they should be — this is what the filter corrects for). The vertical dashed line shows the current threshold; countries to its left are excluded.
- A natural break in the volume distribution (a gap or inflection in the percentile curve) is the principled place to set the threshold.


In [ ]:
# Threshold diagnostic 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Peak citation volume per country across all institutions and blocks (incoming)
peak_vol_by_country = (
    hhi_incoming_df
    .groupby('country_name')['total_citations']
    .max()
    .sort_values(ascending=False)
    .reset_index(name='peak_citations')
)

# Table: how many countries survive at each candidate threshold
thresholds = [100, 500, 1_000, 2_000, 5_000, 10_000, 20_000]
total = len(peak_vol_by_country)
print(f"Total unique countries in dataset: {total}\n")
print(f"{'Threshold':>12}  {'Countries kept':>15}  {'% of total':>12}  {'Excluded':>10}")
print("-" * 56)
for t in thresholds:
    kept = (peak_vol_by_country['peak_citations'] >= t).sum()
    excl = total - kept
    marker = "  ← current" if t == MIN_CITATIONS_HHI else ""
    print(f"{t:>12,}  {kept:>15,}  {kept/total*100:>11.1f}%  {excl:>10,}{marker}")

# Percentile distribution
print("\nPercentile distribution of peak citation volume:")
for p in [10, 25, 50, 75, 90, 95, 99]:
    val = np.percentile(peak_vol_by_country['peak_citations'], p)
    print(f"  p{p:02d}: {val:>10,.0f}")

# Scatterplot: peak volume vs peak HHI
peak_hhi_by_country = (
    hhi_incoming_df
    .groupby('country_name')['HHI']
    .max()
    .reset_index(name='peak_hhi')
)
diag_df = peak_vol_by_country.merge(peak_hhi_by_country, on='country_name')

fig_diag = go.Figure()

# All countries
fig_diag.add_trace(go.Scatter(
    x=diag_df['peak_citations'],
    y=diag_df['peak_hhi'],
    mode='markers',
    marker=dict(
        size=6,
        color=diag_df['peak_citations'],
        colorscale='Viridis',
        showscale=False,
        opacity=0.6,
    ),
    text=diag_df['country_name'],
    hovertemplate='<b>%{text}</b><br>Peak volume: %{x:,.0f}<br>Peak HHI: %{y:,.0f}<extra></extra>',
    name='Countries',
))

# Threshold line
fig_diag.add_vline(
    x=MIN_CITATIONS_HHI,
    line_dash='dash', line_color='crimson', opacity=0.7,
)
fig_diag.add_annotation(
    x=MIN_CITATIONS_HHI, y=10500,
    text=f'MIN_CITATIONS = {MIN_CITATIONS_HHI:,}',
    showarrow=False, xanchor='left',
    font=dict(color='crimson', size=11),
)

fig_diag.update_layout(
    title='<b>Peak Citation Volume vs. Peak HHI — all countries (incoming)</b><br>'
          '<sup>Countries left of the dashed line are excluded from the HHI heatmap</sup>',
    xaxis=dict(
        title='Peak citation volume (log scale)',
        type='log',
        tickvals=[10, 100, 500, 1_000, 5_000, 10_000, 50_000, 100_000, 500_000],
        ticktext=['10', '100', '500', '1k', '5k', '10k', '50k', '100k', '500k'],
        showgrid=True, gridcolor='#ebebeb',
    ),
    yaxis=dict(
        title='Peak HHI',
        showgrid=True, gridcolor='#ebebeb',
    ),
    template='plotly_white',
    height=480,
    margin=dict(t=80, b=60),
)
fig_diag.show()
